# ModernFloraBERT: plant LM -> maize MLM -> regression

This notebook is intended to be opened in VS Code while the notebook kernel is a Google Colab runtime. It does not use Kaggle compute.

The controlled ablation is:

- baseline: the exact plant-pretrained ModernBERT checkpoint -> existing mean-pooling regression model;
- adaptation: the same plant-pretrained ModernBERT checkpoint -> continued MLM on maize promoters -> the same regression model.

The tokenizer is downloaded from Kaggle dataset version 3 and is never retrained. The Kaggle dataset is large, so the notebook records the verified artifact manifest and downloads only the exact version-3 checkpoint/tokenizer and NAM data files it uses. Set `FLORABERT_DOWNLOAD_FULL_KAGGLE_ARCHIVE=1` before running if a full archive is specifically required.

Run cells top-to-bottom. Set `FLORABERT_DRIVE_ROOT` before connecting if downloads and checkpoints should live on mounted Google Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}

# Portable runtime paths. Nothing below assumes /kaggle.
drive_root = os.environ.get('FLORABERT_DRIVE_ROOT', '').strip()
if drive_root:
    if drive_root.startswith('/content/drive'):
        try:
            from google.colab import drive
            if not Path('/content/drive/MyDrive').exists():
                drive.mount('/content/drive')
        except ImportError as exc:
            raise RuntimeError('FLORABERT_DRIVE_ROOT was set, but this is not a Colab runtime') from exc
    run_root = Path(drive_root).expanduser()
else:
    run_root = Path(os.environ.get('FLORABERT_RUN_ROOT', '/content/florabert_runs')).expanduser()

repo_dir = Path(os.environ.get('FLORABERT_REPO_DIR', '/content/florabert')).expanduser()
repo_url = os.environ.get('FLORABERT_REPO_URL', 'https://github.com/gurveervirk/florabert.git')
repo_ref = os.environ.get('FLORABERT_REPO_REF', 'feat/modernbert-maize-mlm-ablation')
# Optionally pin a commit; by default the cloned feature-branch tip is used.
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '')

if not (repo_dir / '.git').is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(f'{repo_dir} exists but is not a clean git checkout')
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', repo_ref, '--depth', '1', repo_url, str(repo_dir)], check=True)

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=repo_dir, text=True
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; expected {expected_commit}. '
        'Override FLORABERT_REPO_COMMIT only when intentionally changing the ablation code.'
    )

run_root.mkdir(parents=True, exist_ok=True)
data_root = run_root / 'data'
kaggle_root = run_root / 'kaggle-modernflorabert-base-v3'
hf_maize_dir = data_root / 'maize-promoter-sequences'
genex_dir = data_root / 'genex' / 'nam'
plant_checkpoint = kaggle_root / 'plant-checkpoint-6000'
plant_tokenizer = kaggle_root / 'modernbert-tokenizer'
model_root = run_root / 'models'
maize_lm_output = model_root / 'transformer' / 'language-model-modernbert-maize'
plant_regression_output = run_root / 'prediction-model-modernbert-plant'
maize_regression_output = run_root / 'prediction-model-modernbert-maize'
for path in [data_root, kaggle_root, hf_maize_dir, genex_dir, plant_checkpoint, plant_tokenizer, maize_lm_output, plant_regression_output, maize_regression_output]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print('Repo:', repo_dir)
print('Repo commit:', actual_commit)
print('Run root:', run_root)
print('CUDA visibility will be checked after dependencies are installed.')

In [ ]:
# Install the repo's actual runtime dependencies into the connected Colab kernel.
# The datasets package is used by module/florabert/dataio.py; Kaggle packages are
# used only for the official versioned artifact API and optional interactive login.
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_dir / 'requirements.txt')],
    cwd=str(repo_dir),
)
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=1.0'],
)

import importlib.metadata as importlib_metadata
for package_name in ['torch', 'transformers', 'datasets', 'accelerate', 'huggingface-hub', 'kagglehub']:
    try:
        print(package_name, importlib_metadata.version(package_name))
    except importlib_metadata.PackageNotFoundError:
        print(package_name, 'not found')


In [ ]:
# Authentication is intentionally credential-free in the notebook source.
# Public HF files can download without a token. For private/gated repos, set
# HF_TOKEN or run interactive_hf_login() in a separate cell.
from huggingface_hub import hf_hub_download

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if hf_token:
    from huggingface_hub import login as hf_login
    hf_login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authentication loaded from an environment variable.')
else:
    print('No HF token found; the requested maize-promoter repository is expected to be public.')

def interactive_hf_login():
    from huggingface_hub import notebook_login
    notebook_login()

# Kaggle authentication is resolved in the remote runtime, not from the local
# VS Code machine. Prefer a Colab Secret or KAGGLE_API_TOKEN; otherwise getpass
# prompts without echoing or writing the token into this notebook.
def load_kaggle_api_token():
    token = os.environ.get('KAGGLE_API_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('KAGGLE_API_TOKEN')
        except Exception:
            token = None
    if token and token.strip():
        os.environ['KAGGLE_API_TOKEN'] = token.strip()
        return True
    return False

has_kaggle_env = load_kaggle_api_token() or any(os.environ.get(name) for name in ['KAGGLE_KEY'])
kaggle_config = Path.home() / '.kaggle' / 'kaggle.json'
kaggle_token_file = Path.home() / '.kaggle' / 'access_token'
if not (has_kaggle_env or kaggle_config.exists() or kaggle_token_file.exists()):
    from getpass import getpass
    entered_token = getpass('Kaggle API token (input hidden; not saved in notebook): ')
    if not entered_token.strip():
        raise RuntimeError('No Kaggle API token was entered.')
    os.environ['KAGGLE_API_TOKEN'] = entered_token.strip()
    del entered_token

import kagglehub
print('KaggleHub authentication is ready.')

In [ ]:
# Download the exact maize MLM files requested by the experiment.
hf_dataset = 'Gurveer05/maize-promoter-sequences'
maize_files = ['all_seqs_train.txt', 'all_seqs_test.txt']

def count_nonempty_lines(path):
    count = 0
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                count += 1
    return count

maize_paths = {}
for filename in maize_files:
    hf_hub_download(
        repo_id=hf_dataset,
        filename=filename,
        repo_type='dataset',
        local_dir=str(hf_maize_dir),
        token=hf_token,
    )
    path = hf_maize_dir / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty HF file: {path}')
    maize_paths[filename] = path
    print(filename, '->', path, 'bytes=', path.stat().st_size, 'sequences=', count_nonempty_lines(path))

assert count_nonempty_lines(maize_paths['all_seqs_train.txt']) > 0
assert count_nonempty_lines(maize_paths['all_seqs_test.txt']) > 0

In [ ]:
# Download the selected version-3 Kaggle artifacts with kagglehub.
# kagglehub uses /versions/3 and supports single-file downloads. The exact
# manifest was verified when the dataset was inspected; using it here avoids
# the separate metadata-listing endpoint that can return 403 from Colab.
import pandas as pd
from IPython.display import display
import kagglehub

kaggle_dataset = 'gurveersinghvirk/modernflorabert-base/versions/3'
required_kaggle_files = [
    'florabert/models/transformer/language-model-modernbert/checkpoint-6000/config.json',
    'florabert/models/transformer/language-model-modernbert/checkpoint-6000/model.safetensors',
    'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer.json',
    'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer_config.json',
    'florabert/models/modernbert-byte-level-bpe-tokenizer/special_tokens_map.json',
    'florabert/data/final/transformer/genex/nam/train.tsv',
    'florabert/data/final/transformer/genex/nam/eval.tsv',
    'florabert/data/final/transformer/genex/nam/test.tsv',
]
kaggle_file_table = pd.DataFrame({
    'name': required_kaggle_files,
    'handle': [kaggle_dataset] * len(required_kaggle_files),
})
print('Version-3 Kaggle manifest:', len(kaggle_file_table), 'required files')
display(kaggle_file_table.to_string(index=False))

kagglehub_stage_dir = kaggle_root / '.kagglehub-files'
kagglehub_stage_dir.mkdir(parents=True, exist_ok=True)
force_kaggle_download = env_bool('FLORABERT_FORCE_KAGGLE_DOWNLOAD', False)

def download_kaggle_file(remote_name, destination_dir):
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    local_path = destination_dir / Path(remote_name).name
    if local_path.is_file() and local_path.stat().st_size > 0 and not force_kaggle_download:
        print('Reusing', local_path, 'bytes=', local_path.stat().st_size)
        return local_path

    downloaded_path = Path(kagglehub.dataset_download(
        kaggle_dataset,
        path=remote_name,
        output_dir=str(kagglehub_stage_dir),
        force_download=force_kaggle_download,
    ))
    source_candidates = [downloaded_path, kagglehub_stage_dir / remote_name]
    source_path = next((path for path in source_candidates if path.is_file()), None)
    if source_path is None or source_path.stat().st_size == 0:
        raise FileNotFoundError(
            f'kagglehub did not produce a non-empty file for {remote_name}; '
            f'returned {downloaded_path}'
        )
    shutil.copy2(source_path, local_path)
    if not local_path.is_file() or local_path.stat().st_size == 0:
        raise IOError(f'Incomplete kagglehub download: {local_path}')
    print(remote_name, '->', local_path, 'bytes=', local_path.stat().st_size)
    return local_path

download_specs = [
    ('florabert/models/transformer/language-model-modernbert/checkpoint-6000/config.json', plant_checkpoint),
    ('florabert/models/transformer/language-model-modernbert/checkpoint-6000/model.safetensors', plant_checkpoint),
    ('florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer.json', plant_tokenizer),
    ('florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer_config.json', plant_tokenizer),
    ('florabert/models/modernbert-byte-level-bpe-tokenizer/special_tokens_map.json', plant_tokenizer),
    ('florabert/data/final/transformer/genex/nam/train.tsv', genex_dir),
    ('florabert/data/final/transformer/genex/nam/eval.tsv', genex_dir),
    ('florabert/data/final/transformer/genex/nam/test.tsv', genex_dir),
]
for remote_name, destination_dir in download_specs:
    download_kaggle_file(remote_name, destination_dir)

if env_bool('FLORABERT_DOWNLOAD_FULL_KAGGLE_ARCHIVE', False):
    full_kaggle_dir = run_root / 'kaggle-modernflorabert-base-v3-full'
    full_kaggle_dir.mkdir(parents=True, exist_ok=True)
    resolved_full = kagglehub.dataset_download(
        kaggle_dataset,
        output_dir=str(full_kaggle_dir),
        force_download=force_kaggle_download,
    )
    print('Full version-3 archive downloaded to', resolved_full)
else:
    print('Full Kaggle archive download skipped; required version-3 artifacts were downloaded individually.')

In [ ]:
# Inspect the selected plant checkpoint and tokenizer before any training.
from transformers import AutoConfig, PreTrainedTokenizerFast

plant_config = AutoConfig.from_pretrained(str(plant_checkpoint), local_files_only=True)
plant_tokenizer_obj = PreTrainedTokenizerFast.from_pretrained(str(plant_tokenizer), local_files_only=True)
print('Plant checkpoint:', plant_checkpoint)
print('Plant model weights:', plant_checkpoint / 'model.safetensors')
print('Plant tokenizer:', plant_tokenizer)
print('Architecture:', plant_config.architectures)
print('model_type:', plant_config.model_type)
print('vocab_size:', plant_config.vocab_size, 'tokenizer length:', len(plant_tokenizer_obj))
print('max_position_embeddings:', plant_config.max_position_embeddings)
print('special token ids:', {name: getattr(plant_tokenizer_obj, name, None) for name in ['cls_token_id', 'sep_token_id', 'pad_token_id', 'unk_token_id', 'mask_token_id']})

assert plant_config.model_type == 'modernbert'
assert plant_config.vocab_size == len(plant_tokenizer_obj), 'Refusing to resize the plant tokenizer vocabulary.'
assert plant_config.architectures == ['ModernBertForMaskedLM']
assert (plant_checkpoint / 'model.safetensors').is_file()
assert all(getattr(plant_config, name, None) == getattr(plant_tokenizer_obj, name, None) for name in ['pad_token_id', 'bos_token_id', 'eos_token_id', 'cls_token_id', 'sep_token_id'])

In [ ]:
# Load the actual repo ModernBERT API and run a strict pretrained-load smoke test.
import torch
from module.florabert import config as flora_config
from module.florabert import transformers as flora_transformers
from module.florabert import utils as flora_utils

flora_config.reload_settings()
lm_settings = flora_utils.get_model_settings(flora_config.settings, model_name='modernbert-lm')
_, loaded_tokenizer, plant_lm = flora_transformers.load_model(
    'modernbert-lm',
    str(plant_tokenizer),
    pretrained_model=str(plant_checkpoint),
    **lm_settings,
)
print('Loaded pretrained checkpoint:', plant_checkpoint)
print('Loaded parameter count:', flora_utils.count_model_parameters(plant_lm, trainable_only=False))
assert len(loaded_tokenizer) == plant_config.vocab_size

smoke_inputs = loaded_tokenizer(
    'ACGTACGTACGTTTTAAACCCGGG',
    return_tensors='pt',
    max_length=loaded_tokenizer.model_max_length,
    truncation=True,
    padding='max_length',
)
plant_lm.eval()
with torch.no_grad():
    smoke_outputs = plant_lm(**smoke_inputs)
assert smoke_outputs.logits.ndim == 3
assert torch.isfinite(smoke_outputs.logits).all()
print('One MLM forward pass:', tuple(smoke_outputs.logits.shape))
del plant_lm, loaded_tokenizer, smoke_outputs, smoke_inputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Keep the first ablation close to the repo's current recipe. In particular,
# do not replace the repo MLM learning rate with the paper's 10x larger value.
pretrain_settings = dict(flora_config.settings['training']['pretrain'])
finetune_settings = dict(flora_config.settings['training']['finetune'])
print('Current repo MLM settings:', pretrain_settings)
print('Current repo regression settings:', finetune_settings)

# None means use config.yaml exactly. These are optional environment overrides.
mlm_learning_rate = float(os.environ['FLORABERT_MLM_LEARNING_RATE']) if os.environ.get('FLORABERT_MLM_LEARNING_RATE') else None
mlm_epochs = int(os.environ['FLORABERT_MLM_EPOCHS']) if os.environ.get('FLORABERT_MLM_EPOCHS') else None
regression_learning_rate = float(os.environ['FLORABERT_REGRESSION_LEARNING_RATE']) if os.environ.get('FLORABERT_REGRESSION_LEARNING_RATE') else None
regression_epochs = int(os.environ['FLORABERT_REGRESSION_EPOCHS']) if os.environ.get('FLORABERT_REGRESSION_EPOCHS') else None
n_workers = int(os.environ.get('FLORABERT_DATA_WORKERS', '2'))
cuda_count = torch.cuda.device_count()
training_precision = 'fp16' if cuda_count else 'no'
print('CUDA devices:', cuda_count)
print('Effective MLM learning rate:', mlm_learning_rate or pretrain_settings['learning_rate'])
print('Effective regression learning rate:', regression_learning_rate or finetune_settings['learning_rate'])
print('Natural-log offset:', finetune_settings.get('log_offset', 0.001))

run_plant_baseline = env_bool('FLORABERT_RUN_PLANT_BASELINE', True)
run_maize_mlm = env_bool('FLORABERT_RUN_MAIZE_MLM', True)
run_maize_regression = env_bool('FLORABERT_RUN_MAIZE_REGRESSION', True)
force_rerun = env_bool('FLORABERT_FORCE_RERUN', False)
mlm_resume_from = Path(os.environ['FLORABERT_MLM_RESUME_FROM']).expanduser() if os.environ.get('FLORABERT_MLM_RESUME_FROM') else None
plant_regression_resume_from = Path(os.environ['FLORABERT_PLANT_REGRESSION_RESUME_FROM']).expanduser() if os.environ.get('FLORABERT_PLANT_REGRESSION_RESUME_FROM') else None
maize_regression_resume_from = Path(os.environ['FLORABERT_MAIZE_REGRESSION_RESUME_FROM']).expanduser() if os.environ.get('FLORABERT_MAIZE_REGRESSION_RESUME_FROM') else None


In [ ]:
# Use one process on a normal single-GPU Colab runtime. Only launch DDP when
# more than one CUDA device is actually visible.
def launch_repo_script(relative_script, arguments):
    script_path = repo_dir / relative_script
    if cuda_count > 1:
        accelerate_exe = shutil.which('accelerate')
        if accelerate_exe:
            command = [accelerate_exe, 'launch', '--num_processes', str(cuda_count), str(script_path), *map(str, arguments)]
        else:
            command = [sys.executable, '-m', 'accelerate.commands.launch', '--num_processes', str(cuda_count), str(script_path), *map(str, arguments)]
    else:
        command = [sys.executable, '-u', str(script_path), *map(str, arguments)]
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(repo_dir) + os.pathsep + environment.get('PYTHONPATH', '')
    # The child process does not have a TTY under Jupyter, so force live
    # stdout/stderr; otherwise startup/tokenization logs can stay buffered
    # for minutes and make an active run look stalled.
    environment['PYTHONUNBUFFERED'] = '1'
    print('Running:', ' '.join(map(str, command)))
    subprocess.run(command, cwd=str(repo_dir), env=environment, check=True)

def append_optional(arguments, flag, value):
    if value is not None:
        arguments.extend([flag, str(value)])

def regression_arguments(output_dir, pretrained_model=None, resume_from=None):
    arguments = [
        '--model-name', 'modernbert-pred-mean-pool',
        '--data-dir', str(genex_dir),
        '--train-data', 'train.tsv',
        '--eval-data', 'eval.tsv',
        '--test-data', 'test.tsv',
        '--tokenizer-dir', str(plant_tokenizer),
        '--output-dir', str(output_dir),
        '--transformation', 'log',
        '--log-offset', '0.001',
        '--precision', training_precision,
        '--n-workers', str(n_workers),
    ]
    if pretrained_model is not None:
        arguments.extend(['--pretrained-model', str(pretrained_model)])
    if resume_from is not None:
        arguments.extend(['--resume-from-checkpoint', str(resume_from)])
    append_optional(arguments, '--learning-rate', regression_learning_rate)
    append_optional(arguments, '--num-train-epochs', regression_epochs)
    return arguments

## Baseline: plant ModernBERT -> regression

This is the clean ablation control. It uses the same plant checkpoint, tokenizer, data, natural-log target, optimizer settings, and regression script as the maize-adapted run.

In [ ]:
if run_plant_baseline:
    baseline_best = plant_regression_output / 'best' / 'config.json'
    if baseline_best.is_file() and not force_rerun and plant_regression_resume_from is None:
        print('Plant baseline already exists; set FLORABERT_FORCE_RERUN=1 to retrain:', plant_regression_output)
    else:
        if not (plant_checkpoint / 'model.safetensors').is_file():
            raise FileNotFoundError(f'Plant checkpoint is missing: {plant_checkpoint}')
        launch_repo_script(
            Path('scripts/1-modeling/finetune.py'),
            regression_arguments(
                plant_regression_output,
                pretrained_model=plant_checkpoint if plant_regression_resume_from is None else None,
                resume_from=plant_regression_resume_from,
            ),
        )
else:
    print('Plant baseline disabled by FLORABERT_RUN_PLANT_BASELINE.')

## Continued MLM: plant ModernBERT -> maize promoters

The source checkpoint is always the plant-pretrained ModernBERT model unless an explicit MLM resume checkpoint is configured. The repo pretrainer uses the current `config.yaml` MLM recipe by default (`learning_rate: 1e-4`, `mlm_prob: 0.15`, LAMB, linear schedule).

In [ ]:
assert maize_paths['all_seqs_train.txt'].is_file() and maize_paths['all_seqs_train.txt'].stat().st_size > 0
assert maize_paths['all_seqs_test.txt'].is_file() and maize_paths['all_seqs_test.txt'].stat().st_size > 0
print('Resolved maize train:', maize_paths['all_seqs_train.txt'])
print('Resolved maize test:', maize_paths['all_seqs_test.txt'])
print('Maize train sequences:', count_nonempty_lines(maize_paths['all_seqs_train.txt']))
print('Maize test sequences:', count_nonempty_lines(maize_paths['all_seqs_test.txt']))
print('Resolved plant checkpoint:', plant_checkpoint)
print('Resolved maize MLM output:', maize_lm_output)

if run_maize_mlm:
    maize_model_config = maize_lm_output / 'config.json'
    if maize_model_config.is_file() and not force_rerun and mlm_resume_from is None:
        print('Maize MLM output already exists; set FLORABERT_FORCE_RERUN=1 to retrain:', maize_lm_output)
    else:
        mlm_arguments = [
            '--model-name', 'modernbert-lm',
            '--data-dir', str(hf_maize_dir),
            '--train-data', 'all_seqs_train.txt',
            '--test-data', 'all_seqs_test.txt',
            '--tokenizer-dir', str(plant_tokenizer),
            '--output-dir', str(maize_lm_output),
            '--precision', training_precision,
            '--n-workers', str(n_workers),
        ]
        mlm_source = mlm_resume_from or plant_checkpoint
        mlm_arguments.extend(['--pretrained-model', str(mlm_source)])
        if mlm_resume_from is not None:
            mlm_arguments.extend(['--resume-from-checkpoint', str(mlm_resume_from)])
        append_optional(mlm_arguments, '--learning-rate', mlm_learning_rate)
        append_optional(mlm_arguments, '--num-train-epochs', mlm_epochs)
        launch_repo_script(Path('scripts/1-modeling/pretrain.py'), mlm_arguments)
else:
    print('Maize MLM disabled by FLORABERT_RUN_MAIZE_MLM.')

In [ ]:
# pretrain.py saves the best Trainer model at its output root. Regression consumes
# this adapted root, never the plant checkpoint or the regression output.
maize_adapted_checkpoint = maize_lm_output
if run_maize_regression and maize_regression_resume_from is None:
    if not (maize_adapted_checkpoint / 'config.json').is_file():
        raise FileNotFoundError(
            f'Maize-adapted MLM checkpoint is not available at {maize_adapted_checkpoint}. '
            'Run the continued-MLM cell first.'
        )
    if not any((maize_adapted_checkpoint / filename).is_file() for filename in ['model.safetensors', 'pytorch_model.bin']):
        raise FileNotFoundError(f'Maize-adapted checkpoint has no model weights: {maize_adapted_checkpoint}')
    print('Exact adapted checkpoint consumed by regression:', maize_adapted_checkpoint)
elif run_maize_regression:
    print('Regression will resume from:', maize_regression_resume_from)
else:
    print('Maize regression is disabled; no adapted checkpoint is required.')

## Regression: maize-adapted ModernBERT -> mean-pooling gene-expression model

Validation is evaluated after every epoch by `finetune.py`. The script writes `metrics.jsonl`, saves every `epoch_*`, and retains the lowest-validation-MSE checkpoint as `best`. The `final` directory is not used for reporting.

In [ ]:
assert all((genex_dir / filename).is_file() and (genex_dir / filename).stat().st_size > 0 for filename in ['train.tsv', 'eval.tsv', 'test.tsv'])
print('Resolved regression data:', genex_dir)
print('Resolved regression output:', maize_regression_output)
if run_maize_regression:
    adapted_best = maize_regression_output / 'best' / 'config.json'
    if adapted_best.is_file() and not force_rerun and maize_regression_resume_from is None:
        print('Maize regression output already exists; set FLORABERT_FORCE_RERUN=1 to retrain:', maize_regression_output)
    else:
        launch_repo_script(
            Path('scripts/1-modeling/finetune.py'),
            regression_arguments(
                maize_regression_output,
                pretrained_model=maize_adapted_checkpoint if maize_regression_resume_from is None else None,
                resume_from=maize_regression_resume_from,
            ),
        )
else:
    print('Maize regression disabled by FLORABERT_RUN_MAIZE_REGRESSION.')

In [ ]:
# Show the validation history and prove that a best checkpoint was retained.
def show_validation_history(output_dir):
    metrics_file = Path(output_dir) / 'metrics.jsonl'
    best_file = Path(output_dir) / 'best_metrics.json'
    if not metrics_file.is_file():
        print('No validation history:', metrics_file)
        return
    records = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
    history = pd.DataFrame([
        {'epoch': record['epoch'], **record['overall']} for record in records
    ])
    display(history[['epoch', 'mse', 'r2', 'pearson_r2', 'prediction_mean', 'prediction_std', 'target_mean', 'target_std']])
    if not best_file.is_file() or not (Path(output_dir) / 'best' / 'config.json').is_file():
        raise RuntimeError(f'Best validation checkpoint missing under {output_dir}')
    print('Best checkpoint metadata:', json.loads(best_file.read_text()))

if run_plant_baseline:
    show_validation_history(plant_regression_output)
show_validation_history(maize_regression_output)

In [ ]:
# Final test evaluation for the retained best checkpoints. Values are in the
# current natural-log target space: ln(TPM + 0.001).
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import r2_score
from module.florabert import dataio as flora_dataio

def report_metrics(targets, predictions, label):
    targets = targets.astype('float64')
    predictions = predictions.astype('float64')
    def one_scope(scope, y_true, y_pred):
        y_true = y_true.reshape(-1)
        y_pred = y_pred.reshape(-1)
        correlation = float('nan') if y_true.std() == 0 or y_pred.std() == 0 else float(np.corrcoef(y_true, y_pred)[0, 1])
        return {
            'scope': scope,
            'mse': float(((y_true - y_pred) ** 2).mean()),
            'sklearn_r2': float(r2_score(y_true, y_pred)),
            'pearson_r2': correlation ** 2 if correlation == correlation else float('nan'),
            'prediction_mean': float(y_pred.mean()),
            'prediction_std': float(y_pred.std()),
            'target_mean': float(y_true.mean()),
            'target_std': float(y_true.std()),
        }
    rows = [one_scope('overall', targets, predictions)]
    for tissue_idx in range(targets.shape[1]):
        tissue = flora_config.tissues[tissue_idx] if tissue_idx < len(flora_config.tissues) else f'tissue_{tissue_idx}'
        rows.append(one_scope(tissue, targets[:, tissue_idx], predictions[:, tissue_idx]))
    result = pd.DataFrame(rows).set_index('scope')
    print(label)
    display(result)
    evaluation_dir = run_root / 'evaluation'
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    (evaluation_dir / f'{label}.json').write_text(json.dumps(rows, indent=2))
    return result

def evaluate_best_checkpoint(output_dir, label):
    checkpoint = Path(output_dir) / 'best'
    if not (checkpoint / 'config.json').is_file():
        raise FileNotFoundError(f'Best checkpoint missing: {checkpoint}')
    pred_settings = flora_utils.get_model_settings(flora_config.settings, model_name='modernbert-pred-mean-pool')
    pred_settings['output_mode'] = 'regression'
    pred_settings['num_labels'] = len(flora_config.tissues)
    _, tokenizer, model = flora_transformers.load_model(
        'modernbert-pred-mean-pool',
        str(plant_tokenizer),
        pretrained_model=str(checkpoint),
        log_offset=0.001,
        **pred_settings,
    )
    test_datasets = flora_dataio.load_datasets(
        tokenizer,
        str(genex_dir / 'test.tsv'),
        seq_key='sequence',
        file_type='csv',
        delimiter='\t',
        transformation='log',
        log_offset=0.001,
        shuffle=False,
        n_workers=n_workers,
    )
    test_dataset = test_datasets['train'].remove_columns(['sequence'])
    loader = DataLoader(test_dataset, batch_size=8, collate_fn=flora_dataio.load_data_collator('pred'), shuffle=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device).eval()
    predictions = []
    targets = []
    with torch.no_grad():
        for batch in loader:
            labels = batch['labels'].to(device)
            inputs = {key: value.to(device) for key, value in batch.items() if key in {'input_ids', 'attention_mask', 'position_ids', 'labels'}}
            outputs = model(**inputs)
            predictions.append(outputs.logits.detach().cpu())
            targets.append(labels.detach().cpu())
    result = report_metrics(torch.cat(targets).numpy(), torch.cat(predictions).numpy(), label)
    del model, tokenizer, test_dataset, test_datasets
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

evaluation_results = {}
if run_plant_baseline:
    evaluation_results['plant_baseline'] = evaluate_best_checkpoint(plant_regression_output, 'plant_baseline')
if run_maize_regression:
    evaluation_results['maize_adapted'] = evaluate_best_checkpoint(maize_regression_output, 'maize_adapted')
else:
    print('Maize regression is disabled; skipping final maize evaluation.')

## Reproducibility and fresh-session checklist

1. In VS Code, select the Colab-connected Python kernel. Optionally set `FLORABERT_DRIVE_ROOT=/content/drive/MyDrive/florabert_runs` before running the setup cell.
2. Run setup, dependency installation, authentication, HF download, and Kaggle metadata/artifact cells.
3. Run the strict checkpoint smoke test. It must print the 5000-token vocabulary match, loaded base tensors, parameter count, and one finite MLM forward pass.
4. Run the baseline cell, the maize MLM cell, then the maize regression cell. To resume MLM, set `FLORABERT_MLM_RESUME_FROM` to a Transformers `checkpoint-*` directory. To resume regression, set the relevant `*_REGRESSION_RESUME_FROM` to an `epoch_*` directory containing `training_state.pt`.
5. Run the history and final test-evaluation cells. Compare `plant_baseline` and `maize_adapted`; use the reported `best` checkpoints.

The remaining differences from the published FloraBERT pipeline are deliberate: this run uses the existing ModernBERT plant checkpoint from Kaggle dataset version 3, the current repo's smaller MLM recipe and natural-log regression settings, the HF maize promoter files requested here, and the current NAM train/eval/test files. It does not retrain the tokenizer, use Kaggle-only paths/compute, or adopt the paper's larger MLM learning rate in the same ablation.